<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/BART_for_ZeroShotClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run this notebook on a GPU runtime (e.g., T4 GPU in Google Colab)

# Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import matplotlib
import os
import pandas
import sys
from transformers import pipeline

In [ ]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

In [ ]:
# from innoprod.digital_readiness_score import DRS_LEVELS
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps

# Notebook settings

In [ ]:
qual_cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

In [ ]:
model_name = "facebook/bart-large-mnli"
classifier = pipeline("zero-shot-classification", model=model_name)

# Retrieve & wrangle data

In [ ]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [ ]:
# TODO: remove the `[:10]` once the workflow is established
qual_df = roadmaps_df[['Client ID'] + qual_cols][:10].copy()
qual_df['Context'] = roadmaps_df.apply(lambda row: ' '.join([row[c] for c in qual_cols]), axis=1)
qual_df.drop(qual_cols, axis=1, inplace=True)
qual_df = qual_df.dropna()
qual_df

In [ ]:
client_ids = qual_df['Client ID'].to_list()
context = qual_df['Context'].to_list()

In [ ]:
questions_prompts = get_sheet_dfs(
    sheet_id="18oFiC2gu6azFaO1XHYFsSpVovBvz08s0hgg97Lu1LGY",
    ranges={"QuestionsPrompts": "Sheet1!A1:G16"}
  )['QuestionsPrompts']
# questions_prompts

In [ ]:
candidate_label_cols = [c for c in questions_prompts.columns.to_list() if c.startswith('Level ')]
candidate_label_cols

# Run pipeline

In [ ]:
all_results_df = None

for idx, row in questions_prompts.iterrows():
  hypothesis_template = (row['Question'] + " {}")
  candidate_labels = [row[c] for c in candidate_label_cols]
  results = classifier(
      context,
      candidate_labels=candidate_labels,
      hypothesis_template=hypothesis_template
    )
  for res in results:
    res['Prediction'] = candidate_labels.index(res['labels'][0]) + 1
    res['Probability'] = res['scores'][0]
    res['Confidence'] = res['scores'][0] - res['scores'][1]
    del res['sequence']
    del res['labels']
    del res['scores']
  results_df = pandas.DataFrame(results)
  results_df['Question'] = row['Question']
  results_df['Client ID'] = client_ids
  if all_results_df is None:
    all_results_df = results_df
  else:
    all_results_df = pandas.concat([all_results_df, results_df])

In [ ]:
questions_prompts['Question'] = pandas.Series(questions_prompts['Question'], dtype="string")
all_results_df['Question'] = pandas.Series(all_results_df['Question'], dtype="string")
all_results_df = pandas.merge(all_results_df, questions_prompts[['Question', 'Core', 'Category']], on="Question", how="left")
all_results_df